In [1]:
import os
%pwd
os.chdir("../")
%pwd

'd:\\Data Science\\END to END Proj\\DrugToxicity'

In [2]:
## ENTITY
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    transformed_data_dir: Path    # Input:  artifacts/data_transformation
    model_bundle_path: Path       # Output: artifacts/model_trainer/tox21_model_bundle.pkl

In [3]:
from src.DrugToxicity.constant import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.DrugToxicity.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer

        create_directories([Path(config.root_dir)])

        return ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            transformed_data_dir=Path(config.transformed_data_dir),
            model_bundle_path=Path(config.model_bundle_path)
        )

In [5]:
import os
import numpy as np
import joblib
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
import xgboost as xgb
import lightgbm as lgb
from src.DrugToxicity.utils.common import logger

TARGET_COLS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

In [6]:
## Model factory functions
def make_rf():
    return RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=2,
        class_weight="balanced", n_jobs=-1, random_state=42
    )

def make_xgb(spw: int):
    return xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric="auc", tree_method="hist",
        random_state=42, n_jobs=-1, verbosity=0
    )

def make_lgbm(spw: int):
    return lgb.LGBMClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw,
        num_leaves=63, min_child_samples=10,
        n_jobs=-1, random_state=42, verbose=-1
    )

def make_meta():
    return LogisticRegression(
        C=1.0, class_weight="balanced", max_iter=1000
    )

In [7]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        """
        Skips if model_bundle_path already exists.
        Otherwise trains XGBoost + RandomForest + LightGBM stacking ensemble:
          - Per-target scale_pos_weight (neg // pos) handles class imbalance
          - 5-fold OOF predictions train the LogisticRegression meta-learner
          - Base models retrained on full training split before saving
          - X_te.npy + y_te.npy saved for model evaluation stage
        Returns True if trained, False if skipped.
        """
        if self.config.model_bundle_path.exists():
            logger.info(
                f"Model bundle already exists at {self.config.model_bundle_path} "
                "— skipping training."
            )
            return False

        os.makedirs(self.config.root_dir, exist_ok=True)

        ## Load features and targets
        X = np.load(self.config.transformed_data_dir / "X.npy")
        y = np.load(self.config.transformed_data_dir / "y.npy")
        logger.info(f"Loaded X{X.shape} y{y.shape}")

        ## Train/test split
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        logger.info(f"Train: {X_tr.shape} | Test: {X_te.shape}")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        model_bundle = {}

        for i, target in enumerate(TARGET_COLS):
            mask_tr = y_tr[:, i] != -1
            X_t = X_tr[mask_tr]
            y_t = y_tr[mask_tr, i]

            if len(np.unique(y_t)) < 2 or len(y_t) < 50:
                logger.warning(f"  {target}: skipped (insufficient labelled data).")
                continue

            pos = int(y_t.sum())
            neg = int((y_t == 0).sum())
            spw = max(1, neg // max(1, pos))   ## per-target imbalance ratio

            logger.info(f"  {target}: {len(y_t)} samples | pos={pos} neg={neg} spw={spw}")

            oof_rf   = np.zeros(len(y_t))
            oof_xgb  = np.zeros(len(y_t))
            oof_lgbm = np.zeros(len(y_t))

            ## 5-fold OOF for meta-learner (no leakage)
            for fold, (tr_idx, val_idx) in enumerate(skf.split(X_t, y_t)):
                Xf_tr, Xf_val = X_t[tr_idx], X_t[val_idx]
                yf_tr         = y_t[tr_idx]

                _rf = make_rf();       _rf.fit(Xf_tr, yf_tr)
                _xg = make_xgb(spw);  _xg.fit(Xf_tr, yf_tr)
                _lg = make_lgbm(spw); _lg.fit(Xf_tr, yf_tr)

                oof_rf[val_idx]   = _rf.predict_proba(Xf_val)[:, 1]
                oof_xgb[val_idx]  = _xg.predict_proba(Xf_val)[:, 1]
                oof_lgbm[val_idx] = _lg.predict_proba(Xf_val)[:, 1]

            ## Train meta-learner on OOF predictions
            meta_X   = np.column_stack([oof_rf, oof_xgb, oof_lgbm])
            meta_clf = make_meta()
            meta_clf.fit(meta_X, y_t)

            ## Retrain base models on full training data
            rf_full   = make_rf();       rf_full.fit(X_t, y_t)
            xgb_full  = make_xgb(spw);  xgb_full.fit(X_t, y_t)
            lgbm_full = make_lgbm(spw); lgbm_full.fit(X_t, y_t)

            model_bundle[target] = {
                "rf"       : rf_full,
                "xgb"      : xgb_full,
                "lgbm"     : lgbm_full,
                "meta"     : meta_clf,
                "spw"      : spw,
                "n_train"  : len(y_t),
                "pos_ratio": pos / len(y_t),
            }
            logger.info(f"  {target}: training complete.")

        ## Save test split for evaluation stage
        np.save(self.config.transformed_data_dir / "X_te.npy", X_te)
        np.save(self.config.transformed_data_dir / "y_te.npy", y_te)

        ## Save full model bundle
        joblib.dump(model_bundle, self.config.model_bundle_path, compress=3)
        size_mb = os.path.getsize(self.config.model_bundle_path) / 1e6
        logger.info(
            f"Model bundle saved → {self.config.model_bundle_path} ({size_mb:.1f} MB)"
        )
        return True

In [8]:
try:
    config_manager = ConfigurationManager()
    trainer_config = config_manager.get_model_trainer_config()
    trainer = ModelTrainer(config=trainer_config)

    if trainer.train():
        logger.info("New models trained and saved.")
    else:
        logger.info("Using existing model bundle.")

except Exception as e:
    logger.error(f"Model training pipeline failed: {str(e)}")
    raise

[2026-03-27 00:24:50,057: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-27 00:24:50,073: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-27 00:24:50,078: INFO: common: created directory at: artifacts]
[2026-03-27 00:24:50,080: INFO: common: created directory at: artifacts\model_trainer]
[2026-03-27 00:24:50,403: INFO: 1243308408: Loaded X(7831, 3133) y(7831, 12)]
[2026-03-27 00:24:51,021: INFO: 1243308408: Train: (6264, 3133) | Test: (1567, 3133)]
[2026-03-27 00:24:51,063: INFO: 1243308408:   NR-AR: 5826 samples | pos=248 neg=5578 spw=22]
[2026-03-27 00:27:35,430: INFO: 1243308408:   NR-AR: training complete.]
[2026-03-27 00:27:35,490: INFO: 1243308408:   NR-AR-LBD: 5405 samples | pos=195 neg=5210 spw=26]
[2026-03-27 00:30:24,184: INFO: 1243308408:   NR-AR-LBD: training complete.]
[2026-03-27 00:30:24,289: INFO: 1243308408:   NR-AhR: 5242 samples | pos=601 neg=4641 spw=7]
[2026-03-27 00:33:47,964: INFO: 1243308408:   NR-AhR: training compl

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\logging\__init__.py", line 1113, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\dell\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2192' in position 63: character maps to <undefined>
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "d:\Data Science\END to END Proj\DrugToxicity\venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "d:\Dat

[2026-03-27 00:56:31,117: INFO: 2834902706: New models trained and saved.]
